# pymefpp_run

This is a wrapper script to the mefpp engine to reduce the data that has been partioned and associated with the scripts

*pymefpp_partition_calibration
*pymefpp_partition_science
*pymefpp_associate

After running those scripts we end up with a  directory named reduced that contains the following tree structure:

* reduced/calibs
        * /bpm
        * /bias
        * /dome
        * /sky
    * /science
    
Below science we have all the science partition to be reduced.

This script will:

* take a science partition at a time
* execute the reduction cascade

The idea is to identify all biases, dome flats, sky flats that are needed for the full reduction of that science partitions, and only these, and reduce them. This permits a piecemeal approach to the work.


        
 

In [1]:
from os import listdir
from os.path import isfile, join
import glob
from astropy.io import fits
import math
import operator
import copy
#import datetime
import os
import shutil
#import subprocess
from datetime import datetime
#from pathlib import Path
from pprint import pprint

# here we import the local mefpp_run library
import sys
sys.path.insert(0,'/home/lceledon/mefpp/python')
from mefpp_run import *
sys.path
#mefpp_run.__file__

['/home/lceledon/mefpp/python',
 '/opt/anaconda3/lib/python36.zip',
 '/opt/anaconda3/lib/python3.6',
 '/opt/anaconda3/lib/python3.6/lib-dynload',
 '',
 '/opt/anaconda3/lib/python3.6/site-packages',
 '/opt/anaconda3/lib/python3.6/site-packages/IPython/extensions',
 '/home/lceledon/.ipython']

In [2]:
# we set the data reduction directories -- LONG TIME TO RUN
# 
raw_data_dir = '/home/lschmidt/fernando/fits/'
root_dir = '/home/lceledon/local_storage/omegacam/testing/'
reduced_dir = root_dir + '/reduced/'
reduced_cal_dir = reduced_dir+'/calibs/'
reduced_science_dir = reduced_dir+'/science/'
log_dir = reduced_dir+'/logs/'

# were to find mefpp module
mefpp_dir = '/home/lceledon/mefpp' 

os.chdir(root_dir)
global cwd
cwd = os.getcwd()
print('cwd: ', cwd)

# depending on size of data set this line might be very slow
science_list = glob.glob(reduced_science_dir+'**/science.list', recursive=True)
print('number of science partitions: ', len(science_list))


cwd:  /homelocal2/lceledon/omegacam/testing
number of science partitions:  205


In [3]:
# associations for the science frame 
sci_partitions = []
for science_file in science_list:
    sci_partitions.append(ExposurePartition(science_file, 'SCIENCE'))


In [4]:
# select data to reduce
print('--------------------- science frame associations -------------------')
science_partition_list = sci_partitions
i = 0
for partition in science_partition_list:
    print(i, partition.base_path)
    i+=1

--------------------- science frame associations -------------------
0 /home/lceledon/local_storage/omegacam/testing//reduced//science/science/CN_Ori_055207p790_m052500p5/r_SDSS/2017-03-11T01:24:08
1 /home/lceledon/local_storage/omegacam/testing//reduced//science/science/CN_Ori_055207p790_m052500p5/r_SDSS/2017-03-12T00:24:41
2 /home/lceledon/local_storage/omegacam/testing//reduced//science/science/CN_Ori_055207p790_m052500p5/NB_659/2017-03-12T00:24:41
3 /home/lceledon/local_storage/omegacam/testing//reduced//science/science/CN_Ori_055207p790_m052500p5/NB_659/2017-03-11T01:24:08
4 /home/lceledon/local_storage/omegacam/testing//reduced//science/science/BX_Pup_075415p580_m241936p5/NB_659/2017-03-11T02:24:50
5 /home/lceledon/local_storage/omegacam/testing//reduced//science/science/BX_Pup_075415p580_m241936p5/NB_659/2017-03-13T02:47:17
6 /home/lceledon/local_storage/omegacam/testing//reduced//science/science/BX_Pup_075415p580_m241936p5/r_SDSS/2017-03-11T02:24:50
7 /home/lceledon/local_stora

In [5]:
# we do the association here
science_partition_list = sci_partitions[68:70]
for scipart in science_partition_list:

    bias_part_list = []
    dome_part_list = []
    sky_part_list = []
    sci_part_list = []

    # -------------- DATA ASSOCIATION SECTION ---------------
    print('scipart: '+scipart.base_path)
    sci_part_list.append(scipart)
    associateBPM(scipart, scipart)
    print('        bpm part:  '+scipart.bpmPartition.base_path)
    associateBias(scipart, scipart)
    print('        bias part:  '+scipart.biasPartition.base_path)
    bias_part_list.append(scipart.biasPartition)
    associateMFF(scipart, scipart)
    print('        MFF  part:  '+scipart.mffPartition.base_path)

    # associations for the MFF frame
    mffpart = scipart.mffPartition
    print('         mffpart: '+mffpart.base_path)
    associateBPM(mffpart, scipart)
    print('                  bpm  part:  '+mffpart.bpmPartition.base_path)
    associateBias(mffpart, scipart)
    print('                  bias part:  '+mffpart.biasPartition.base_path)
    bias_part_list.append(mffpart.biasPartition)
    associateDome(mffpart, scipart)
    print('                  dome part:  '+mffpart.domePartition.base_path)
    dome_part_list.append(mffpart.domePartition)
    associateSky(mffpart, scipart)
    print('                   sky part:  '+mffpart.skyPartition.base_path)
    sky_part_list.append(mffpart.skyPartition)

    # associations for the SKY frame
    skypart = mffpart.skyPartition
    print('  ---> skypart: '+skypart.base_path)
    associateBPM(skypart, scipart)
    print('                  bpm  part:  '+skypart.bpmPartition.base_path)
    associateBias(skypart, scipart)
    print('                  bias part:  '+skypart.biasPartition.base_path)
    bias_part_list.append(skypart.biasPartition)
    associateDome(skypart, scipart)
    print('                  dome part:  '+skypart.domePartition.base_path)
    dome_part_list.append(skypart.domePartition)

    # associations for the dome frame
    domepart = ExposurePartition(skypart.domePartition.frame, 'FLAT,DOME')
    print('  ---> domepart: '+domepart.base_path)
    associateBPM(domepart, scipart)
    print('                  bpm  part:  '+domepart.bpmPartition.base_path)
    associateBias(domepart, scipart)
    print('                  bias part:  '+domepart.biasPartition.base_path)
    bias_part_list.append(domepart.biasPartition)

    # --------------------- DATA REDUCTION SECTION ---------------------
    # here we reduce the bpm frames which for now is only a cp operation
    mefpp_reduce_bpm(mefpp_dir, reduced_cal_dir)

    # we start reducing the biases
    mefpp_reduce_bias(mefpp_dir, bias_part_list)

    # reduction of dome frames
    mefpp_reduce_dome(mefpp_dir, dome_part_list)

    # reduction of sky frames
    mefpp_reduce_sky(mefpp_dir, sky_part_list)

    # we check if the science.list only have one line
    # if does, we add an extra line to avoid failing
    # this extra line correspond to a science object of BX Pup at NB_659 or r_SDSS depending on filter
    oneline = False
    f0 = open(scipart.base_path+'/science.list','r')
    content = f0.readlines()
    f0.close()
    if len(content) == 1:
        oneline = True
        if scipart.filter == 'r_SDSS':
            extra_science = 'OMEGA.2017-03-11T03:15:54.553'
        elif scipart.filter == 'NB_659':
            extra_science = 'OMEGA.2017-03-11T02:40:24.297'
        f0 = open(scipart.base_path+'/science.list','a')
        f0.write('\n/home/lschmidt/fernando/fits/{}.fits'.format(extra_science))
        f0.close()
        
    # we reduce the science frames here
    mefpp_reduce_science(mefpp_dir, sci_part_list)
    
    # We erase the aditional fits image created if the science list have only one line
    # and restore the science.list to his former state
    if oneline:
        os.remove(scipart.base_path + '/{}_pre.fits'.format(extra_science))
        f0 = open(scipart.base_path+'/science.list', 'w')
        f0.write(content[0])
        f0.close()
    # Normalize science. This will show a single face to the
    # procedures that follow-
    #
    #  - divide by EXPTIME
    #  - divide by avg gain: from MFF get gain vs average counts line get gain for counts = 1.0
    # add AIRMASS keyword
    # add PHOT_K keyword
    # add SATURATION keyword
    # add extinction K keyword
    # add PHOT_C keyword
    #
    # mefpp_normalize_science(mefpp, sci_part_list)



scipart: /home/lceledon/local_storage/omegacam/testing//reduced//science/science/LS_IV_-08_3_165629p600_m083439p1/r_SDSS/2017-03-15T09:13:03
        bpm part:  /homelocal2/lceledon/omegacam/testing/reduced/calibs/bpm
        bias part:  /homelocal2/lceledon/omegacam/testing/reduced/calibs/bias/574184/2017-03-15T09:55:35
        MFF  part:  /homelocal2/lceledon/omegacam/testing/reduced/calibs/sky/200253584/r_SDSS/2017-03-13T23:12:13
         mffpart: /homelocal2/lceledon/omegacam/testing/reduced/calibs/sky/200253584/r_SDSS/2017-03-13T23:12:13
                  bpm  part:  /homelocal2/lceledon/omegacam/testing/reduced/calibs/bpm
                  bias part:  /homelocal2/lceledon/omegacam/testing/reduced/calibs/bias/574184/2017-03-14T09:41:34
                  dome part:  /homelocal2/lceledon/omegacam/testing/reduced/calibs/dome/574184/r_SDSS/2017-03-14T09:41:34
                   sky part:  /homelocal2/lceledon/omegacam/testing/reduced/calibs/sky/200253584/r_SDSS/2017-03-13T23:12:13
  --

mefpp_reduce_bpm: done copying BPMs
mefpp_reduce_bias: length of bias partition list: 4
mefpp_reduce_bias:  /homelocal2/lceledon/omegacam/testing/reduced/calibs/bias/574184/2017-04-12T10:10:09/bias.list
basepath:  /homelocal2/lceledon/omegacam/testing/reduced/calibs/bias/574184/2017-04-12T10:10:09
bias_already_reduced:  True
bias_list_exists:  True
mefpp_paf_exists:  True
bpm_exists:  True
mefpp_reduce_bias:  /homelocal2/lceledon/omegacam/testing/reduced/calibs/bias/574184/2017-04-13T09:54:14/bias.list
basepath:  /homelocal2/lceledon/omegacam/testing/reduced/calibs/bias/574184/2017-04-13T09:54:14
bias_already_reduced:  True
bias_list_exists:  True
mefpp_paf_exists:  True
bpm_exists:  True
mefpp_reduce_bias:  /homelocal2/lceledon/omegacam/testing/reduced/calibs/bias/574184/2017-04-07T09:56:42/bias.list
basepath:  /homelocal2/lceledon/omegacam/testing/reduced/calibs/bias/574184/2017-04-07T09:56:42
bias_already_reduced:  True
bias_list_exists:  True
mefpp_paf_exists:  True
bpm_exists:  Tr